# 05 — Supply-Chain Feature Engineering

**Goal:** turn raw Comtrade + WGI into a per-(product, source_country) feature matrix the risk model can train on.

**Output:** `data/processed/risk_features.parquet`, columns:
`product_hs, source_country, distance_km, tariff_pct, trade_value_log,
country_risk, source_concentration, free_zone_flag, historic_disruption_count, y_risk_score`.

`y_risk_score` is our proxy target — a composite of historic disruption count + volatility, min-max scaled into [0,1].

## Setup

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd

SEED = 42
RAW = Path('../data/raw'); PROC = Path('../data/processed'); PROC.mkdir(exist_ok=True, parents=True)

## Load

In [ ]:
trade = pd.read_csv(RAW/'comtrade_electronics.csv')
wgi   = pd.read_csv(RAW/'wgi.csv')
print(trade.shape, wgi.shape)

## Country risk from WGI

Political Stability score, standardised.

In [ ]:
wgi = wgi[wgi.indicator=='PoliticalStability'][['country','2023']].rename(columns={'2023':'wgi_pol'})
wgi['country_risk'] = 1 - (wgi.wgi_pol - wgi.wgi_pol.min()) / (wgi.wgi_pol.max() - wgi.wgi_pol.min())

## Source concentration (HHI)

Higher = fewer countries dominate → riskier if that region disrupts.

In [ ]:
def hhi(g):
    s = g.trade_value_usd
    return ((s/s.sum())**2).sum()
hhi_by_hs = trade.groupby('hs_code').apply(hhi).rename('source_concentration').reset_index()

## Assemble

Join per (hs_code, reporter). Distance uses simple Haversine to primary US port (Long Beach).

In [ ]:
import math
def haversine(lat1, lon1, lat2, lon2):
    R=6371; p1=math.radians(lat1); p2=math.radians(lat2)
    dp=p2-p1; dl=math.radians(lon2-lon1)
    a=math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

# In practice load a country-centroid table; this is illustrative.
centroids = pd.DataFrame({'country':['China','USA','Germany'], 'lat':[35.0, 39.7, 51.1], 'lon':[103, -95, 10]})
LB = (33.7, -118.2)
centroids['distance_km'] = centroids.apply(lambda r: haversine(r.lat, r.lon, *LB), axis=1)

## Target — historic disruption count

Proxy: rolling year-over-year volatility of trade value. High volatility = disrupted.

In [ ]:
vol = trade.groupby(['hs_code','reporter']).trade_value_usd.std().rename('trade_std').reset_index()
target = vol.copy()
target['y_risk_score'] = (target.trade_std - target.trade_std.min()) / (target.trade_std.max()-target.trade_std.min())

## Save

In [ ]:
features = target.rename(columns={'reporter':'source_country','hs_code':'product_hs'})
features.to_parquet(PROC/'risk_features.parquet')
print(features.shape); features.head()